# Notebook 04b: Pose + Left/Right Hand Landmarks & Multimodal Fusion Upgrade

**Why this notebook:** Notebook 03 & 04 showed frozen full-frame ImageNet `ResNet50` features plateau around `~11–12%` Top-1 / `~25.7%` Top-5 across unseen signers because of background/clothing shortcuts.  
**Solution (Python 3.13 & Colab CPU friendly):** Use **`rtmlib` (`Wholebody` lightweight ONNX via `onnxruntime`)** to extract **53 ASL landmarks** (`0..10` upper body + `21` left-hand + `21` right-hand keypoints) → normalized into background-free **`(32, 349)`** coordinates, hand-relative finger shapes, and velocities.

### What This Notebook Does
1. Unpacks `voxel51_wlasl100_videos.tar` cleanly to `/content/wlasl100_cache/videos/` and extracts resumable `(N, 32, 349)` pose+hand sequences (`P_val.npy`, `P_test.npy`, `P_train.npy` = ~50 MB total), saving progress to Drive every `32` clips.
2. Saves a skeleton overlay grid (`pose_hand_check.png`) so you can visually verify hand & upper-body tracking.
3. Trains & benchmarks **on `val` (`test` remains locked until Notebook 05)**:
   - `pose_transformer_only` (Pure Hand+Pose+Motion baseline)
   - `mm_pose_language` (Hand+Pose + `all-MiniLM-L6-v2` prompt alignment)
   - `mm_pose_resnet_language` (Triple-stream Fusion: Hand+Pose + ResNet50 + Language)


## 1. Environment Setup (`rtmlib` + `onnxruntime` for Python 3.13) & Drive Mount


In [ ]:
import importlib
import subprocess
import sys

import torch

ONNX_PKG = "onnxruntime-gpu" if torch.cuda.is_available() else "onnxruntime"
for mod_name, pip_name in [("rtmlib", "rtmlib"), ("onnxruntime", ONNX_PKG), ("cv2", "opencv-python-headless")]:
    try:
        importlib.import_module(mod_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import json
import math
import random
import shutil
import tarfile
import time
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from rtmlib import Wholebody
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB and colab_drive is not None:
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted at /content/drive.")
    else:
        colab_drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ONNX_DEV = "cuda" if DEVICE.type == "cuda" else "cpu"
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | Device: {DEVICE} | ONNX: {ONNX_DEV}")

## 2. Configuration & Fast SSD Video Staging


In [ ]:
# ==============================================================================
# CONFIGURATION & FAST SSD VIDEO STAGING
# ==============================================================================
SEED = 42
NUM_CLASSES = 100
NUM_FRAMES = 32
POSE_DIM = 349
VISUAL_DIM = 2048
FLUSH_EVERY_CLIPS = 64 if DEVICE.type == "cuda" else 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
    SCRATCH_ROOT = Path("/content/wlasl100_cache")
else:
    DRIVE_ROOT = Path.cwd() / "data" / "voxel51_wlasl100"
    SCRATCH_ROOT = Path.cwd() / "data" / "wlasl100_scratch"

META_DIR = DRIVE_ROOT / "metadata"
FEAT_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
PLOTS_DIR = DRIVE_ROOT / "reports" / "plots"
VIS_DIR = DRIVE_ROOT / "reports" / "visual_examples"
TAR_PATH = DRIVE_ROOT / "videos" / "voxel51_wlasl100_videos.tar"

for d in (FEAT_DIR, CKPT_DIR, EXP_DIR, PLOTS_DIR, VIS_DIR, SCRATCH_ROOT):
    d.mkdir(parents=True, exist_ok=True)

LOCAL_VIDEOS = SCRATCH_ROOT / "videos"
LOCAL_VIDEOS.mkdir(parents=True, exist_ok=True)

if len(list(LOCAL_VIDEOS.glob("*.mp4"))) < 1120:
    t0 = time.time()
    print(f"Unpacking {TAR_PATH} -> {LOCAL_VIDEOS} ...")
    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(LOCAL_VIDEOS)
    print(f"Unpacked {len(list(LOCAL_VIDEOS.glob('*.mp4')))} videos in {time.time() - t0:.1f}s")
else:
    print(f"Videos already staged on SSD: {len(list(LOCAL_VIDEOS.glob('*.mp4')))}")


def normalize_video_id(val: Any) -> str:
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return str(int(s)) if s.isdigit() else s


class_to_id = {str(k): int(v) for k, v in json.loads((META_DIR / "class_to_id.json").read_text(encoding="utf-8")).items()}
id_to_class = {int(k): str(v) for k, v in json.loads((META_DIR / "id_to_class.json").read_text(encoding="utf-8")).items()}

splits = {}
for sp in ("train", "val", "test"):
    df = pd.read_csv(META_DIR / f"{sp}.csv")
    df["video_id_str"] = df["video_id"].map(normalize_video_id)
    splits[sp] = df

## 3. Resumable 53-Keypoint Pose + Left/Right Hand Feature Extraction


In [ ]:
# ==============================================================================
# STEP 1: RESUMABLE 53-KEYPOINT (UPPER BODY + 21L + 21R HANDS) EXTRACTION
# ==============================================================================
UPPER_BODY_INDICES = list(range(11))
LEFT_HAND_INDICES = list(range(91, 112))
RIGHT_HAND_INDICES = list(range(112, 133))
ASL_KEYPOINT_INDICES = UPPER_BODY_INDICES + LEFT_HAND_INDICES + RIGHT_HAND_INDICES
NUM_ASL_KEYPOINTS = 53


def parse_bbox_xyxy(bbox_json: Any, width: int, height: int, pad_ratio: float = 0.08):
    if pd.isna(bbox_json) or not str(bbox_json).strip():
        return None
    try:
        box = json.loads(str(bbox_json))
    except Exception:
        return None
    if not isinstance(box, list) or len(box) != 4:
        return None
    x, y, bw, bh = [float(v) for v in box]
    if bw <= 0 or bh <= 0 or width <= 0 or height <= 0:
        return None
    x1 = max(0.0, (x - pad_ratio * bw) * width)
    y1 = max(0.0, (y - pad_ratio * bh) * height)
    x2 = min(float(width), (x + (1.0 + pad_ratio) * bw) * width)
    y2 = min(float(height), (y + (1.0 + pad_ratio) * bh) * height)
    return [x1, y1, x2, y2] if x2 > x1 and y2 > y1 else None


def normalize_asl_keypoints_sequence(raw_kpts: np.ndarray, raw_scores: np.ndarray, width: int, height: int) -> np.ndarray:
    t_len = raw_kpts.shape[0]
    sel_xy = raw_kpts[:, ASL_KEYPOINT_INDICES, :2].astype(np.float32)
    sel_conf = np.clip(raw_scores[:, ASL_KEYPOINT_INDICES], 0.0, 1.0).astype(np.float32)
    norm_xy = np.zeros((t_len, NUM_ASL_KEYPOINTS, 2), dtype=np.float32)
    hand_rel = np.zeros((t_len, 42, 2), dtype=np.float32)

    def_center = np.array([width * 0.5, height * 0.5], dtype=np.float32)
    def_scale = max(float(width) * 0.25, 16.0)

    for t in range(t_len):
        xy, cf = sel_xy[t], sel_conf[t]
        valid = cf >= 0.15
        if valid[5] and valid[6]:
            center = 0.5 * (xy[5] + xy[6])
            scale = max(float(np.linalg.norm(xy[5] - xy[6])), 16.0)
        elif np.any(valid[:11]):
            ub = xy[:11][valid[:11]]
            center = ub.mean(axis=0)
            scale = max(float(np.linalg.norm(ub.max(axis=0) - ub.min(axis=0))) * 0.5, def_scale)
        else:
            center, scale = def_center, def_scale

        scaled = np.clip((xy - center) / scale, -5.0, 5.0)
        scaled[~valid] = 0.0
        norm_xy[t] = scaled

        if valid[11]:
            lh_sc = max(float(np.linalg.norm(xy[20] - xy[11])), 0.25 * scale) if valid[20] else 0.35 * scale
            lh = np.clip((xy[11:32] - xy[11:12]) / max(lh_sc, 8.0), -4.0, 4.0)
            lh[~valid[11:32]] = 0.0
            hand_rel[t, :21] = lh
        if valid[32]:
            rh_sc = max(float(np.linalg.norm(xy[41] - xy[32])), 0.25 * scale) if valid[41] else 0.35 * scale
            rh = np.clip((xy[32:53] - xy[32:33]) / max(rh_sc, 8.0), -4.0, 4.0)
            rh[~valid[32:53]] = 0.0
            hand_rel[t, 21:] = rh

    vel = np.zeros_like(norm_xy)
    if t_len > 1:
        vel[1:] = np.clip(norm_xy[1:] - norm_xy[:-1], -3.0, 3.0)

    out = np.concatenate([norm_xy.reshape(t_len, 106), hand_rel.reshape(t_len, 84), vel.reshape(t_len, 106), sel_conf], axis=-1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)


wb = Wholebody(to_openpose=False, mode="lightweight", backend="onnxruntime", device=ONNX_DEV)


def extract_clip_pose(video_path: Path, bbox_json: Any = None):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ok, frm = cap.read()
        if not ok or frm is None:
            break
        frames.append(frm)
    cap.release()
    assert len(frames) > 0, f"No frames: {video_path}"

    h, w = frames[0].shape[:2]
    idxs = np.round(np.linspace(0, len(frames) - 1, NUM_FRAMES)).astype(int)
    bbox = parse_bbox_xyxy(bbox_json, w, h)
    if bbox is None:
        det_boxes = wb.det_model(frames[idxs[0]])
        bbox_list = [det_boxes[0].tolist()] if len(det_boxes) > 0 else [[0.0, 0.0, float(w), float(h)]]
    else:
        bbox_list = [bbox]

    raw_kpts = np.zeros((NUM_FRAMES, 133, 2), dtype=np.float32)
    raw_scores = np.zeros((NUM_FRAMES, 133), dtype=np.float32)
    sampled_frames = []
    for t_i, f_idx in enumerate(idxs.tolist()):
        frm = frames[f_idx]
        if t_i in (4, 12, 20, 28):
            sampled_frames.append((f_idx, frm))
        kpts, sc = wb.pose_model(frm, bboxes=bbox_list)
        if len(kpts) > 0:
            best = int(np.argmax(sc[:, ASL_KEYPOINT_INDICES].mean(axis=1)))
            raw_kpts[t_i] = kpts[best, :133, :2]
            raw_scores[t_i] = sc[best, :133]

    feats = normalize_asl_keypoints_sequence(raw_kpts, raw_scores, w, h)
    return feats, raw_kpts, raw_scores, sampled_frames


def _sync_pose(sp: str, p_arr: np.ndarray, d_arr: np.ndarray) -> None:
    tmp_p = SCRATCH_ROOT / f"P_{sp}.npy"
    tmp_d = SCRATCH_ROOT / f"done_pose_{sp}.npy"
    np.save(tmp_p, p_arr)
    np.save(tmp_d, d_arr)
    shutil.copy2(tmp_p, FEAT_DIR / f"P_{sp}.npy")
    shutil.copy2(tmp_d, FEAT_DIR / f"done_pose_{sp}.npy")


for sp in ("val", "test", "train"):
    df = splits[sp]
    n_rows = len(df)
    p_drv = FEAT_DIR / f"P_{sp}.npy"
    d_drv = FEAT_DIR / f"done_pose_{sp}.npy"
    if p_drv.is_file() and d_drv.is_file():
        p_prev = np.load(p_drv)
        d_prev = np.load(d_drv)
        if p_prev.shape == (n_rows, NUM_FRAMES, POSE_DIM) and len(d_prev) == n_rows:
            p_arr = np.array(p_prev, dtype=np.float32, copy=True)
            d_arr = np.array(d_prev, dtype=bool, copy=True)
            print(f"Resuming pose [{sp}]: {int(d_arr.sum())}/{n_rows} done.")
        else:
            p_arr = np.zeros((n_rows, NUM_FRAMES, POSE_DIM), dtype=np.float32)
            d_arr = np.zeros(n_rows, dtype=bool)
    else:
        p_arr = np.zeros((n_rows, NUM_FRAMES, POSE_DIM), dtype=np.float32)
        d_arr = np.zeros(n_rows, dtype=bool)

    pending = np.flatnonzero(~d_arr)
    newly_done = 0
    for pos_i, idx in enumerate(tqdm(pending.tolist(), desc=f"Pose+Hands [{sp}]")):
        row = df.iloc[idx]
        vp = LOCAL_VIDEOS / f"{row['video_id_str']}.mp4"
        feats, _, _, _ = extract_clip_pose(vp, row.get("bbox_json"))
        p_arr[idx] = feats
        d_arr[idx] = True
        newly_done += 1
        if newly_done >= FLUSH_EVERY_CLIPS or pos_i == len(pending) - 1:
            _sync_pose(sp, p_arr, d_arr)
            newly_done = 0

    assert bool(d_arr.all()) and np.isfinite(p_arr).all()
    print(f"Pose [{sp:5s}] ready: {p_arr.shape}, upper_conf_mean={p_arr[..., 296:307].mean():.3f}, hand_conf_mean={p_arr[..., 307:].mean():.3f}")

## 4. Visual Verification of Hand & Upper-Body Landmarks


In [ ]:
# ==============================================================================
# STEP 2: VISUAL VERIFICATION OF EXTRACTED HAND & UPPER-BODY LANDMARKS
# ==============================================================================
fig, axes = plt.subplots(4, 4, figsize=(13, 12))
val_sample = splits["val"].sample(n=4, random_state=SEED)
for r_i, (_, row) in enumerate(val_sample.iterrows()):
    vp = LOCAL_VIDEOS / f"{row['video_id_str']}.mp4"
    _, raw_k, raw_s, samp_frames = extract_clip_pose(vp, row.get("bbox_json"))
    frame_map = [4, 12, 20, 28]
    for c_i, (t_orig, frm) in enumerate(samp_frames):
        rgb = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB).copy()
        t_pos = frame_map[c_i]
        for k_idx in UPPER_BODY_INDICES:
            if raw_s[t_pos, k_idx] >= 0.25:
                cx, cy = int(raw_k[t_pos, k_idx, 0]), int(raw_k[t_pos, k_idx, 1])
                cv2.circle(rgb, (cx, cy), 4, (255, 215, 0), -1)
        for k_idx in LEFT_HAND_INDICES:
            if raw_s[t_pos, k_idx] >= 0.25:
                cx, cy = int(raw_k[t_pos, k_idx, 0]), int(raw_k[t_pos, k_idx, 1])
                cv2.circle(rgb, (cx, cy), 3, (0, 255, 0), -1)
        for k_idx in RIGHT_HAND_INDICES:
            if raw_s[t_pos, k_idx] >= 0.25:
                cx, cy = int(raw_k[t_pos, k_idx, 0]), int(raw_k[t_pos, k_idx, 1])
                cv2.circle(rgb, (cx, cy), 3, (0, 200, 255), -1)
        axes[r_i, c_i].imshow(rgb)
        axes[r_i, c_i].axis("off")
        if c_i == 0:
            axes[r_i, c_i].set_title(f"{row['gloss']} (s{row['signer_id']})")
plt.suptitle("Pose + Left Hand (Green) + Right Hand (Cyan) Keypoints Verification", fontsize=13)
plt.tight_layout()
plt.savefig(VIS_DIR / "pose_hand_check.png", dpi=120)
plt.show()

## 5. Train & Benchmark Pose + Visual + Language Models on Validation


In [ ]:
# ==============================================================================
# STEP 3: POSE + VISUAL + LANGUAGE MULTIMODAL MODELS & TRAINING (TRAIN -> VAL)
# ==============================================================================
P_train = np.load(FEAT_DIR / "P_train.npy").astype(np.float32)
P_val = np.load(FEAT_DIR / "P_val.npy").astype(np.float32)
X_train = np.load(FEAT_DIR / "X_train.npy").astype(np.float32)
X_val = np.load(FEAT_DIR / "X_val.npy").astype(np.float32)
y_train = np.load(FEAT_DIR / "y_train.npy").astype(np.int64)
y_val = np.load(FEAT_DIR / "y_val.npy").astype(np.int64)

vocab_prompt = torch.load(FEAT_DIR / "text_embeddings_prompt.pt", map_location="cpu", weights_only=False)["text_embeddings"].float().to(DEVICE)


def augment_pose(seq: np.ndarray, rng=None) -> np.ndarray:
    gen = rng if rng is not None else np.random.default_rng()
    t_len = seq.shape[0]
    ratio = float(gen.uniform(0.85, 1.0))
    win = max(1, min(t_len, int(round(t_len * ratio))))
    st = int(gen.integers(0, t_len - win + 1)) if win < t_len else 0
    idxs = np.round(np.linspace(st, st + win - 1, t_len)).astype(np.int64)
    aug = seq[idxs].astype(np.float32, copy=True)
    aug[:, :296] *= float(gen.uniform(0.90, 1.10))
    shift = gen.normal(0.0, 0.04, size=(1, 1, 2)).astype(np.float32)
    aug[:, :106].reshape(t_len, 53, 2)[:] += shift
    aug[:, :296] += gen.normal(0.0, 0.015, size=(t_len, 296)).astype(np.float32)
    drop = gen.random((t_len, 53)) < 0.05
    aug[:, 296:][drop] = 0.0
    nc = aug[:, :106].reshape(t_len, 53, 2)
    vl = aug[:, 190:296].reshape(t_len, 53, 2)
    vl[0] = 0.0
    vl[1:] = np.clip(nc[1:] - nc[:-1], -3.0, 3.0)
    return aug


def augment_visual(seq: np.ndarray, rng=None) -> np.ndarray:
    gen = rng if rng is not None else np.random.default_rng()
    t_len = seq.shape[0]
    ratio = float(gen.uniform(0.85, 1.0))
    win = max(1, min(t_len, int(round(t_len * ratio))))
    st = int(gen.integers(0, t_len - win + 1)) if win < t_len else 0
    idxs = np.round(np.linspace(st, st + win - 1, t_len)).astype(np.int64)
    aug = seq[idxs].astype(np.float32, copy=True)
    keep = (gen.random((1, aug.shape[1])) < 0.75).astype(np.float32)
    aug = (aug * keep) / 0.75
    return aug + gen.normal(0.0, 0.02, size=aug.shape).astype(np.float32)


class PoseVisualDataset(Dataset):
    def __init__(self, p_arr, x_arr, y_arr, augment=False):
        self.p, self.x, self.y, self.augment = p_arr, x_arr, y_arr, augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        p_seq = np.array(self.p[idx], dtype=np.float32, copy=True)
        x_seq = np.array(self.x[idx], dtype=np.float32, copy=True)
        if self.augment:
            p_seq = augment_pose(p_seq)
            x_seq = augment_visual(x_seq)
        return torch.from_numpy(p_seq), torch.from_numpy(x_seq), torch.tensor(int(self.y[idx]), dtype=torch.long)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        return (seq * torch.softmax(self.proj(seq), dim=1)).sum(dim=1)


class TemporalTransformerEncoder(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int = 192, num_layers: int = 2, num_heads: int = 4, dim_feedforward: int = 384, dropout: float = 0.35, max_seq_len: int = 128, pooling: str = "attention"):
        super().__init__()
        self.input_proj = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, embed_dim), nn.GELU(), nn.Dropout(dropout))
        pe = torch.zeros(1, max_seq_len, embed_dim)
        pos = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-math.log(10000.0) / embed_dim))
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pos_encoding", pe, persistent=False)
        enc_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=dim_feedforward, dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.attn_pool = TemporalAttentionPool(embed_dim)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        t_len = features.size(1)
        x = self.input_proj(features) + self.pos_encoding[:, :t_len, :]
        return self.attn_pool(self.norm(self.encoder(x)))


class TextProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 384, embed_dim: int = 192, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(input_dim, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class PoseVisualMultimodalModel(nn.Module):
    def __init__(self, num_classes: int = 100, *, pose_dim: int = 349, visual_dim: int = 2048, text_dim: int = 384, hidden_dim: int = 192, embed_dim: int = 192, num_layers: int = 2, num_heads: int = 4, dropout: float = 0.35, mode: str = "pose_visual_language", fusion_alpha: float = 0.5, init_temperature: float = 0.1):
        super().__init__()
        self.mode = mode
        self.fusion_alpha = fusion_alpha
        self.pose_encoder = TemporalTransformerEncoder(pose_dim, hidden_dim, num_layers, num_heads, hidden_dim * 2, dropout)
        if mode == "pose_visual_language":
            self.visual_encoder = TemporalTransformerEncoder(visual_dim, hidden_dim, num_layers, num_heads, hidden_dim * 2, dropout)
            self.stream_gate = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.Sigmoid())
        else:
            self.visual_encoder, self.stream_gate = None, None
        self.motion_projection = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.classifier_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))
        if mode in {"pose_language", "pose_visual_language"}:
            self.text_projection = TextProjectionHead(text_dim, embed_dim, dropout=0.15)
            self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / init_temperature), dtype=torch.float32))
            self.lang_fusion_gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
            self.fused_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(embed_dim, num_classes))
        else:
            self.text_projection, self.logit_scale, self.lang_fusion_gate, self.fused_head = None, None, None, None

    def forward(self, pose_inputs: torch.Tensor, *, visual_inputs: torch.Tensor | None = None, vocab_text_embeddings: torch.Tensor | None = None):
        pose_repr = self.pose_encoder(pose_inputs)
        if self.mode == "pose_visual_language":
            vis_repr = self.visual_encoder(visual_inputs)
            g_s = self.stream_gate(torch.cat([pose_repr, vis_repr], dim=-1))
            repr_vec = g_s * pose_repr + (1.0 - g_s) * vis_repr
        else:
            repr_vec = pose_repr
        if self.mode == "pose_only":
            return {"logits": self.classifier_head(repr_vec)}
        vis_emb = F.normalize(self.motion_projection(repr_vec), p=2, dim=-1)
        txt_emb = self.text_projection(vocab_text_embeddings)
        scale = self.logit_scale.exp().clamp(max=30.0)
        cosine_logits = scale * (vis_emb @ txt_emb.T)
        attn = torch.softmax(cosine_logits, dim=-1)
        lang_ctx = attn @ txt_emb
        g_l = self.lang_fusion_gate(torch.cat([vis_emb, lang_ctx], dim=-1))
        fused = g_l * vis_emb + (1.0 - g_l) * lang_ctx
        cls_logits = self.fused_head(fused) + self.fusion_alpha * cosine_logits
        return {"logits": cls_logits, "cosine_logits": cosine_logits, "visual_embeddings": vis_emb, "text_embeddings": txt_emb, "logit_scale": scale}


def compute_metrics(y_true: np.ndarray, logits: np.ndarray) -> dict[str, Any]:
    y_t = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(logits, dtype=np.float64)
    y_pred = np.argmax(scores, axis=1)
    top1 = float(np.mean(y_pred == y_t))
    top5 = float(np.mean(np.any(np.argpartition(-scores, kth=4, axis=1)[:, :5] == y_t[:, None], axis=1)))
    f1s = []
    for cid in range(scores.shape[1]):
        tp = int(np.sum((y_t == cid) & (y_pred == cid)))
        fp = int(np.sum((y_t != cid) & (y_pred == cid)))
        fn = int(np.sum((y_t == cid) & (y_pred != cid)))
        if tp + fn == 0:
            continue
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn)
        f1s.append((2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0)
    return {"top1_accuracy": top1, "top5_accuracy": top5, "macro_f1": float(np.mean(f1s)), "unique_predicted_classes": int(len(np.unique(y_pred)))}


def compute_retrieval(sim: np.ndarray, q_lbl: np.ndarray, g_lbl: np.ndarray) -> dict[str, float]:
    order = np.argsort(-sim, axis=1)
    ranks = np.asarray([int(np.flatnonzero(g_lbl[o] == q_lbl[i])[0]) + 1 for i, o in enumerate(order)], dtype=np.float64)
    return {"mrr": float(np.mean(1.0 / ranks)), "recall@1": float(np.mean(ranks <= 1)), "recall@5": float(np.mean(ranks <= 5))}


def multi_pos_infonce(v: torch.Tensor, t: torch.Tensor, s: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    lg = s * (v @ t.T)
    same = (y.unsqueeze(1) == y.unsqueeze(0)).float()
    tgt = same / same.sum(dim=1, keepdim=True).clamp(min=1.0)
    return 0.5 * (-(tgt * F.log_softmax(lg, dim=1)).sum(dim=1).mean() - (tgt.T * F.log_softmax(lg.T, dim=1)).sum(dim=1).mean())


def run_pose_exp(exp_id: str, mode: str, *, hidden_dim: int = 192, dropout: float = 0.35, lr: float = 8e-4, weight_decay: float = 5e-3, epochs: int = 80, patience: int = 25):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = PoseVisualMultimodalModel(NUM_CLASSES, pose_dim=POSE_DIM, visual_dim=VISUAL_DIM, text_dim=384, hidden_dim=hidden_dim, embed_dim=hidden_dim, dropout=dropout, mode=mode).to(DEVICE)
    run_dir = CKPT_DIR / exp_id
    run_dir.mkdir(parents=True, exist_ok=True)

    counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float64)
    w_s = (1.0 / np.sqrt(np.maximum(counts, 1.0)))[y_train]
    sampler = WeightedRandomSampler(weights=w_s.tolist(), num_samples=len(y_train), replacement=True)
    tr_loader = DataLoader(PoseVisualDataset(P_train, X_train, y_train, augment=True), batch_size=32, sampler=sampler)
    va_loader = DataLoader(PoseVisualDataset(P_val, X_val, y_val, augment=False), batch_size=32, shuffle=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (ep + 1) / 5.0 if ep < 5 else 0.5 * (1.0 + math.cos(math.pi * (ep - 5) / max(1, epochs - 5))))

    best_score, best_ep, best_m, no_imp, hist = (-1.0, float("-inf")), 0, {}, 0, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tr_l, tr_n = 0.0, 0
        for pb, xb, yb in tr_loader:
            pb, xb, yb = pb.to(DEVICE), xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(pb, visual_inputs=xb if mode == "pose_visual_language" else None, vocab_text_embeddings=None if mode == "pose_only" else vocab_prompt)
            if mode == "pose_only":
                loss = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
            else:
                ce = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
                vc = F.cross_entropy(out["cosine_logits"], yb, label_smoothing=0.08)
                inf = multi_pos_infonce(out["visual_embeddings"], out["text_embeddings"][yb], out["logit_scale"], yb)
                loss = 0.5 * ce + 0.5 * vc + 0.25 * inf
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_l += float(loss.detach()) * yb.size(0); tr_n += yb.size(0)
        sched.step()

        model.eval()
        va_l, va_n, lg_all, vis_all, lbl_all, txt_p = 0.0, 0, [], [], [], None
        with torch.inference_mode():
            for pb, xb, yb in va_loader:
                pb, xb, yb = pb.to(DEVICE), xb.to(DEVICE), yb.to(DEVICE)
                out = model(pb, visual_inputs=xb if mode == "pose_visual_language" else None, vocab_text_embeddings=None if mode == "pose_only" else vocab_prompt)
                loss = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
                va_l += float(loss.detach()) * yb.size(0); va_n += yb.size(0)
                lg_all.append(out["logits"].cpu().numpy()); lbl_all.append(yb.cpu().numpy())
                if mode != "pose_only":
                    vis_all.append(out["visual_embeddings"].cpu().numpy()); txt_p = out["text_embeddings"].cpu().numpy()

        lg_np, lbl_np = np.concatenate(lg_all), np.concatenate(lbl_all)
        vm = compute_metrics(lbl_np, lg_np)
        vm["loss"] = va_l / max(1, va_n)
        if mode != "pose_only" and txt_p is not None:
            sim = np.concatenate(vis_all) @ txt_p.T
            rv2t = compute_retrieval(sim, lbl_np, np.arange(NUM_CLASSES))
            rt2v = compute_retrieval(sim.T, np.arange(NUM_CLASSES), lbl_np)
            vm["v2t_recall@1"], vm["v2t_recall@5"], vm["v2t_mrr"], vm["t2v_recall@1"] = rv2t["recall@1"], rv2t["recall@5"], rv2t["mrr"], rt2v["recall@1"]

        hist.append({"epoch": ep, "train_loss": tr_l / tr_n, "val_loss": vm["loss"], "val_top1": vm["top1_accuracy"], "val_top5": vm["top5_accuracy"], "val_macro_f1": vm["macro_f1"]})
        sc = (vm["top1_accuracy"], -vm["loss"])
        if sc > best_score:
            best_score, best_ep, best_m, no_imp = sc, ep, dict(vm), 0
            ckpt = {"epoch": ep, "model_state_dict": model.state_dict(), "val_metrics": best_m, "class_to_id": class_to_id, "id_to_class": id_to_class, "config": {"pose_dim": POSE_DIM, "visual_dim": VISUAL_DIM, "hidden_dim": hidden_dim, "embed_dim": hidden_dim, "dropout": dropout, "mode": mode}}
            if mode != "pose_only":
                ckpt["vocab_text_embeddings"] = vocab_prompt.detach().cpu()
            torch.save(ckpt, run_dir / "best_model.pt")
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    pd.DataFrame(hist).to_csv(run_dir / "training_history.csv", index=False)
    r5 = f" | V2T_R@5={best_m['v2t_recall@5']*100:5.2f}%" if mode != "pose_only" else ""
    print(f"[{exp_id:28s}] best_ep={best_ep:2d} | val_loss={best_m['loss']:.3f} | val_top1={best_m['top1_accuracy']*100:5.2f}% | val_top5={best_m['top5_accuracy']*100:5.2f}% | f1={best_m['macro_f1']:.4f}{r5} | uniq={best_m['unique_predicted_classes']:3d}/100 | {time.time()-t0:.1f}s")
    return {"experiment_id": exp_id, "mode": mode, "history": hist, "val_metrics": best_m, "ckpt_path": str(run_dir / "best_model.pt")}


r_pose_only = run_pose_exp("pose_transformer_only", "pose_only", hidden_dim=192, dropout=0.35)
r_mm_pose = run_pose_exp("mm_pose_language", "pose_language", hidden_dim=192, dropout=0.35)
r_mm_triple = run_pose_exp("mm_pose_resnet_language", "pose_visual_language", hidden_dim=192, dropout=0.40)

## 6. Curves, Results Tracker & Final Summary Output


In [ ]:
# ==============================================================================
# STEP 4: COMPARE CURVES, UPDATE RESULTS TRACKER & PRINT FINAL SUMMARY
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for res, lbl in [
    (r_pose_only, "Pose+Hands Only"),
    (r_mm_pose, "MM Pose+Hands+Language"),
    (r_mm_triple, "MM Pose+Hands+ResNet50+Lang"),
]:
    hdf = pd.DataFrame(res["history"])
    axes[0].plot(hdf["epoch"], hdf["val_loss"], label=lbl)
    axes[1].plot(hdf["epoch"], hdf["val_top1"] * 100.0, label=lbl)
    axes[2].plot(hdf["epoch"], hdf["val_top5"] * 100.0, label=lbl)
axes[0].set_title("Validation Loss")
axes[1].set_title("Validation Top-1 Accuracy (%)")
axes[2].set_title("Validation Top-5 Accuracy (%)")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plot_path = PLOTS_DIR / "04b_pose_multimodal_curves.png"
plt.savefig(plot_path, dpi=130)
plt.show()

summary_rows = []
for res, mdl_name, mod_name in [
    (r_pose_only, "PoseHandsTransformer(192)", "pose_only"),
    (r_mm_pose, "MM-PoseHands+MiniLM", "multimodal_pose"),
    (r_mm_triple, "MM-PoseHands+ResNet50+MiniLM", "multimodal_triple"),
]:
    vm = res["val_metrics"]
    summary_rows.append({
        "experiment_id": res["experiment_id"],
        "model": mdl_name,
        "modality": mod_name,
        "fusion_mode": res["mode"],
        "split": "val",
        "top1_accuracy": round(vm["top1_accuracy"], 4),
        "top5_accuracy": round(vm["top5_accuracy"], 4),
        "macro_f1": round(vm["macro_f1"], 4),
        "v2t_recall@1": round(vm["v2t_recall@1"], 4) if "v2t_recall@1" in vm else np.nan,
        "v2t_recall@5": round(vm["v2t_recall@5"], 4) if "v2t_recall@5" in vm else np.nan,
        "t2v_recall@1": round(vm["t2v_recall@1"], 4) if "t2v_recall@1" in vm else np.nan,
        "mrr": round(vm["v2t_mrr"], 4) if "v2t_mrr" in vm else np.nan,
        "checkpoint_path": res["ckpt_path"],
    })

new_df = pd.DataFrame(summary_rows)
res_csv = EXP_DIR / "results_summary.csv"
if res_csv.is_file():
    old_df = pd.read_csv(res_csv)
    old_df = old_df[~((old_df["experiment_id"].isin(new_df["experiment_id"])) & (old_df["split"] == "val"))]
    frames = [df.dropna(axis=1, how="all") for df in (old_df, new_df) if not df.empty]
    merged_df = pd.concat(frames, ignore_index=True)
else:
    merged_df = new_df
merged_df.to_csv(res_csv, index=False)

print("=" * 72)
print("NOTEBOOK 04b (POSE+HANDS + MULTIMODAL UPGRADE ON VAL) COMPLETED")
print("=" * 72)
print(new_df[["experiment_id", "mode" if "mode" in new_df else "fusion_mode", "top1_accuracy", "top5_accuracy", "macro_f1", "v2t_recall@1", "v2t_recall@5", "mrr"]].to_string(index=False))
print(f"\nSaved pose visual grid: {VIS_DIR / 'pose_hand_check.png'}")
print(f"Saved curves plot     : {plot_path}")
print(f"Updated results CSV   : {res_csv}")